# Multi-saccades-estimation

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import random
import copy

import subprocess

from urllib.request import urlopen

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from torchvision import datasets, transforms

from pathlib import Path

from torchvision.models import ResNet50_Weights
from torchvision.transforms import functional as FF

from s3c.models.foveated_vit import FoveatedMultiViT, build_foveated_pos_embed
from s3c.models.heads import ShiftPredictor, FovealSetTransformer
from s3c.models.student_teacher import StudentWithYPredictor
from s3c.data.datasets import FoveatedUpletDataset, make_dataloader, collate_uplet
from s3c.data.transforms import ShiftZoomUplet, FoveatedPyramidTransform
from s3c.utils.viz import foveate_mosaic, compute_saliency_grid
from s3c.models.heads import IterativeSeedTransformerwithQuery, AttentionPooling, ABMILPosPredictor 

import timm

from PIL import Image

from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device('cpu')

In [3]:
# Monter le dossier distant
local=True
if local == False:
    mount_point = os.path.expanduser("~/imagenet_mount")

    try:
        os.makedirs(mount_point, exist_ok=True)
        subprocess.run(["sshfs", "dauce.e@brain-lid-004:data/Imagenet_full", mount_point, "-o", "reconnect"], check=True)
    except:
        pass

    train_dir = os.path.join(mount_point, "train")
    val_dir = os.path.join(mount_point, "val")
else:
    #train_dir = "/media/dauce/T7/data/Imagenet_full/train" #
    train_dir = "~/data/Imagenet_full/train"   # Imagenet Validation set
    #val_dir = "/media/dauce/T7/data/Imagenet_full/val" #
    val_dir = "~/data/Imagenet_full/val"   # Imagenet Validation set

In [4]:
# ImageFolder sans transform — le Dataset s'en charge entièrement
train_dataset_raw = datasets.ImageFolder(train_dir, transform=None)
val_dataset_raw   = datasets.ImageFolder(val_dir,   transform=None)

uplet_tf = ShiftZoomUplet(zoom=1.5, std=0.3, n_uplet=5)

batch_size=1
num_workers = 12

zoom = 1.5

std = 0.5 / zoom

n_sac = 5



In [5]:
def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """tensor : (C, H, W) ou (B, C, H, W)"""
    m = torch.tensor(mean).view(-1, 1, 1)
    s = torch.tensor(std).view(-1, 1, 1)
    return (tensor.cpu().float() * s + m).clamp(0, 1)

## Teacher / Student

In [10]:
model_dir = "../checkpoints/checkpoints_EMA_Xattn_260416"

epoch_teacher = 20



model_name = "vit_base_patch8_224.dino"
model_orig = timm.create_model(model_name, pretrained=True)

#model.head = nn.Identity()  # Supprimer la tête existante
model_orig.patch_embed.strict_img_size = False
norm_ref = copy.deepcopy(model_orig.norm)

build_foveated_pos_embed(model_orig,
                        base_img_size=224,
                        patch_size=8,
                        mosaic_sub_patch_grid=8,   # 8 patches par sous-image (64px/8)
                        scales=(1.0, 0.5, 0.25, 0.125))


Parameter containing:
tensor([[[-0.0104, -0.0027, -0.0067,  ..., -0.0113,  0.0012,  0.0053],
         [ 0.0073, -0.0007,  0.0055,  ...,  0.0012, -0.0004,  0.0041],
         [ 0.0029, -0.0116, -0.0105,  ..., -0.0010, -0.0013,  0.0027],
         ...,
         [-0.0103,  0.0063, -0.0050,  ...,  0.0066,  0.0003, -0.0142],
         [-0.0095,  0.0028, -0.0048,  ...,  0.0049, -0.0003, -0.0121],
         [-0.0091,  0.0008, -0.0048,  ...,  0.0040, -0.0006, -0.0108]]],
       requires_grad=True)

In [11]:
teacher_model = timm.create_model(model_name, pretrained=True)
teacher_model.patch_embed.strict_img_size = False
teacher_model.norm = nn.Identity() # no final layernorm!!

build_foveated_pos_embed(teacher_model,
                        base_img_size=224,
                        patch_size=8,
                        mosaic_sub_patch_grid=8,   # 8 patches par sous-image (64px/8)
                        scales=(1.0, 0.5, 0.25, 0.125))


checkpoint_path = os.path.join(model_dir, f"checkpoint_epoch{epoch_teacher}.pt")  # exemple
checkpoint = torch.load(checkpoint_path, map_location="cpu")
# Vérifie les clés disponibles
print("Clés du checkpoint :", checkpoint.keys())
# --- Récupération des poids du teacher ---
if "teacher" not in checkpoint:
    raise KeyError(f"Aucune clé 'teacher' trouvée dans {checkpoint_path}")

state_dict = checkpoint["teacher"]

# --- Chargement dans le modèle ---
missing, unexpected = teacher_model.load_state_dict(state_dict, strict=False)

print("➡️ Poids chargés (teacher).")
print("❗ Paramètres manquants :", missing)
print("⚠️ Paramètres inattendus :", unexpected)

teacher_model.to(device)
teacher_model.eval()


Clés du checkpoint : dict_keys(['epoch', 'batch', 'student', 'teacher', 'mlp', 'history'])
➡️ Poids chargés (teacher).
❗ Paramètres manquants : []
⚠️ Paramètres inattendus : []


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(8, 8), stride=(8, 8))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
  

In [12]:
checkpoint.keys()

dict_keys(['epoch', 'batch', 'student', 'teacher', 'mlp', 'history'])

In [13]:
dino_student_y = StudentWithYPredictor(model_orig).to(device)

if "student" not in checkpoint:
    raise KeyError(f"Aucune clé 'student' trouvée dans {checkpoint_path}")

state_dict = checkpoint["student"]
# --- Chargement dans le modèle ---
missing, unexpected = dino_student_y.load_state_dict(state_dict, strict=False)
print("➡️ Poids chargés (student).")
print("❗ Paramètres manquants :", missing)
print("⚠️ Paramètres inattendus :", unexpected)
dino_student_y = dino_student_y.to(device) #copy.deepcopy(dino_model).to(device)

dino_student_y.eval()

➡️ Poids chargés (student).
❗ Paramètres manquants : []
⚠️ Paramètres inattendus : []


StudentWithYPredictor(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(8, 8), stride=(8, 8))
    (norm): Identity()
  )
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
  (blocks): ModuleList(
    (0-11): 12 x ViTBlockWithYFull(
      (norm1_x): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (norm2_x): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn_x): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (mlp_x): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (fc2): Linear(in_featu

In [14]:
mlp = ShiftPredictor(emb_dim=768, hidden_dim=512)

state_dict = checkpoint["mlp"]

# --- Chargement dans le modèle ---
missing, unexpected = mlp.load_state_dict(state_dict, strict=False)

print("➡️ Poids chargés (mlp).")
print("❗ Paramètres manquants :", missing)
print("⚠️ Paramètres inattendus :", unexpected)

mlp.to(device)
mlp.eval()

➡️ Poids chargés (mlp).
❗ Paramètres manquants : []
⚠️ Paramètres inattendus : []


ShiftPredictor(
  (net): Sequential(
    (0): Linear(in_features=1536, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)

In [15]:
norm_ref = copy.deepcopy(model_orig.norm)
norm_ref.to(device)
norm_ref.train()

LayerNorm((768,), eps=1e-06, elementwise_affine=True)

In [16]:
#z_linear_head_dir = "./checkpoints_260319_EMA_Xattn_1_view"
z_linear_head_dir = "../checkpoints/checkpoints_260414_EMA_Xattn_1_view"

z_linear_head = nn.Linear(model_orig.embed_dim, 1000)

checkpoint_path = os.path.join(z_linear_head_dir, f"checkpoint_epoch20.pt")  
checkpoint = torch.load(checkpoint_path, map_location="cpu")
# Vérifie les clés disponibles
print("Clés du checkpoint :", checkpoint.keys())
# --- Récupération des poids du teacher ---
if "classifier" not in checkpoint:
    raise KeyError(f"Aucune clé 'classifier' trouvée dans {checkpoint_path}")

state_dict = checkpoint["classifier"]

# --- Chargement dans le modèle ---
missing, unexpected = z_linear_head.load_state_dict(state_dict, strict=False)

print("➡️ Poids chargés (mlp).")
print("❗ Paramètres manquants :", missing)
print("⚠️ Paramètres inattendus :", unexpected)

z_linear_head.to(device)
z_linear_head.train()

Clés du checkpoint : dict_keys(['epoch', 'history', 'classifier'])
➡️ Poids chargés (mlp).
❗ Paramètres manquants : []
⚠️ Paramètres inattendus : []


Linear(in_features=768, out_features=1000, bias=True)

In [17]:
resolution = 128

transform_test = transforms.Compose([
    transforms.Resize(512),
    transforms.CenterCrop(512),
    FoveatedPyramidTransform(output_size=resolution, to_torch=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [18]:
def load_and_transform(path, transform):
    img = Image.open(path).convert("RGB")
    return transform(img)  # retourne un (3, H, W) torch.Tensor


## Multi-saccades test

In [19]:
@torch.no_grad()
def compute_norm_grid(
    dino_student_y,
    img1,
    grid_size=11,
    y_min=-5/6, y_max=5/6,
    device='cuda'
):
    device = next(dino_student_y.parameters()).device
    img1   = img1.to(device)

    # ── normalisation batch dim ───────────────────────────────────────
    if img1.ndim == 3:
        img1 = img1.unsqueeze(0)          # (1, C, H, W)
    B = img1.shape[0]

    # ── grille de positions ───────────────────────────────────────────
    xs       = torch.linspace(y_min, y_max, grid_size, device=device)
    ys       = torch.linspace(y_min, y_max, grid_size, device=device)
    X, Y     = torch.meshgrid(xs, ys, indexing="xy")
    grid     = torch.stack([X, Y], dim=-1)     # (G, G, 2)
    grid_flat = grid.reshape(-1, 2)            # (G*G, 2)
    G2       = grid_flat.shape[0]              # G*G = 121

    # ── répéter images et positions pour forward vectorisé ───────────
    # img1     : (B, C, H, W) → (B*G*G, C, H, W)
    # grid_flat: (G*G, 2)     → (B*G*G, 2)
    img1_rep  = img1.unsqueeze(1).expand(B, G2, -1, -1, -1) \
                    .reshape(B * G2, *img1.shape[1:])         # (B*G*G, C, H, W)
    grid_rep  = grid_flat.unsqueeze(0).expand(B, G2, 2) \
                         .reshape(B * G2, 2)                  # (B*G*G, 2)

    # ── forward ──────────────────────────────────────────────────────
    _, z_y = dino_student_y(img1_rep, grid_rep, layernorm=False)  # (B*G*G, D)

    # ── reshape → (B, G, G, D) ───────────────────────────────────────
    D      = z_y.shape[-1]
    z_y    = z_y.reshape(B, grid_size, grid_size, D)              # (B, G, G, D)

    # ── saliency = norme L2 par position ─────────────────────────────
    saliency = torch.linalg.norm(z_y, dim=-1)                     # (B, G, G)

    # ── position maximale par image ───────────────────────────────────
    # argmax sur la grille aplatie
    flat_idx = saliency.reshape(B, -1).argmax(dim=1)              # (B,)
    i_star   = flat_idx // grid_size                              # ligne   (B,)
    j_star   = flat_idx  % grid_size                              # colonne (B,)
    x_star   = xs[j_star]                                         # (B,) — axe x = colonnes
    y_star   = ys[i_star]                                         # (B,) — axe y = lignes

    return {
        "grid"     : grid,                    # (G, G, 2)
        "z_y"      : z_y,                     # (B, G, G, D)
        "saliency" : saliency,                # (B, G, G)
        "i_star"   : i_star,                  # (B,) indices ligne
        "j_star"   : j_star,                  # (B,) indices colonne
        "x_star"   : x_star,                  # (B,) coordonnée x optimale
        "y_star"   : y_star,                  # (B,) coordonnée y optimale
    }


     # (B,)

In [20]:
def shift_zoom(img, x, y, zoom):
        w, h = img.size
        size_ref = min(w, h)
        w_zoomed   = int(w * zoom)
        h_zoomed   = int(h * zoom)
        zoomed_ref = min(w_zoomed, h_zoomed)   # petit côté zoomé — référence du shift

        img_zoomed = FF.resize(img, (h_zoomed, w_zoomed),
                            interpolation=Image.BICUBIC)
        y_prim = y

        # centre de fixation dans l'espace zoomé
        # x,y ∈ [-1,1] définis par rapport au petit côté original
        cx = w_zoomed / 2 + zoomed_ref * x / 2 #zoomed_ref / 2 * (1 + x / 2)   # ∈ [zoomed_ref/4, 3*zoomed_ref/4]
        cy = h_zoomed / 2 + zoomed_ref * y_prim / 2 #zoomed_ref / 2 * (1 + y / 2)

        # crop de taille originale (w, h) centré en (cx, cy)
        left   = int(cx - size_ref/2) #w / 2)
        top    = int(cy - size_ref/2) #h / 2)
        right  = left + size_ref #w
        bottom = top  + size_ref #h

        # padding reflect uniquement si on dépasse l'image zoomée réelle
        pl = max(0, -left);             
        pt = max(0, -top)
        pr = max(0, right  - w_zoomed); 
        pb = max(0, bottom - h_zoomed)

        if any([pl, pt, pr, pb]):
            img_zoomed = FF.pad(img_zoomed, (pl, pt, pr, pb),
                                padding_mode='reflect')
        left += pl
        top  += pt

        return img_zoomed.crop((left, top, left + size_ref, top + size_ref))

In [21]:
def collate_pil(batch):
    """Retourne une liste de PIL Images et un tenseur de labels."""
    imgs   = [b[0] for b in batch]   # liste de PIL Images
    labels = torch.tensor([b[1] for b in batch])
    return imgs, labels

In [ ]:
@torch.no_grad()
def evaluate_saccade(
    dino_student_y,
    z_linear_head,
    val_dataset_raw,
    fovea_transform,
    zoom       = 1.5,
    grid_size  = 11,
    batch_size = 32,
    device     = 'cuda',
    num_workers = 4,
    disp = False
):

    dino_student_y.eval()
    z_linear_head.eval()

    correct_center  = 0
    correct_saccade = 0
    total           = 0

    loader = DataLoader(
        val_dataset_raw,
        batch_size  = batch_size,
        shuffle     = True,
        num_workers = num_workers,
        collate_fn  = collate_pil,   
    )

    pbar = tqdm(loader, desc="Évaluation saccade")

    for i, (imgs_pil, labels) in enumerate(pbar):
        # imgs_pil : liste de B PIL images (collate par défaut)
        labels = labels.to(device)
        B      = labels.shape[0]

        # ── 1. vue centrale (x=0, y=0) ───────────────────────────────────
        mosaics_center = []
        for img in imgs_pil:
            img_center  = shift_zoom(img, 0.0, 0.0, zoom)
            mosaic     = fovea_transform(img_center)
            mosaics_center.append(mosaic)

        mosaics_center = torch.stack(mosa01_teacher_one_saccade_estimation.ipynbics_center).to(device)  # (B, 3, H, W)
        print(mosaics_center.shape)

        # forward student — vue centrale
        z_center = teacher_model(mosaics_center)             # (B, D)
        print(z_center.shape)

        logits_center = z_linear_head(z_center)                  # (B, 1000)
        print(logits_center.shape)
        pred_center   = logits_center.argmax(dim=1)
        correct_center += (pred_center == labels).sum().item()

        # ── 2. compute_norm_grid → position optimale ──────────────────────
        print(f"compute norm grid")
        result = compute_norm_grid(
            dino_student_y,
            mosaics_center,
            grid_size = grid_size,
            device    = device,
        )

        plt.figure(figsize= (8, 4*B))
        for b,img in enumerate(imgs_pil):
            img_center  = shift_zoom(img, 0.0, 0.0, zoom)
            mosaic     = fovea_transform(img_center)
            if disp:
                img_denorm = mosaic.cpu() * torch.tensor(std) + torch.tensor(mean)
                img_fov = foveate_mosaic(img_denorm)
                img_fov = img_fov.permute(0, 2, 3, 1).numpy()
                img_fov = np.clip(img_fov, 0, 1)
                plt.subplot(B, 2, b*2 + 1)
                plt.imshow(img_fov[0,:], extent=coords)
                plt.imshow(result["saliency"][b,:,:].detach().cpu().numpy(), extent=coords, alpha=0.5, cmap='jet')
                plt.plot(result["x_star"][b], -result["y_star"][b],'+w', markersize=10)    

        x_star = result["x_star"]/zoom   # (B,)
        y_star = result["y_star"]/zoom   # (B,)

        # ── 3. vue saccadée (x_star, y_star) ─────────────────────────────
        mosaics_saccade = []
        for b, img in enumerate(imgs_pil):
            print(f"image {b}/{batch_size} : ({x_star[b].item()}, {y_star[b].item()})")
            img_shift = shift_zoom(img,
                                   x_star[b].item(),
                                   y_star[b].item(),
                                   zoom)
            mosaic    = fovea_transform(img_shift)
            mosaics_saccade.append(mosaic)
            if disp:
                img_denorm = mosaic.cpu() * torch.tensor(std) + torch.tensor(mean)
                img_fov = foveate_mosaic(img_denorm)
                img_fov = img_fov.permute(0, 2, 3, 1).numpy()
                img_fov = np.clip(img_fov, 0, 1)
                plt.subplot(B, 2, b*2 + 2)
                plt.imshow(img_fov[0,:], extent=coords)
                plt.plot(0,0,'+w', markersize=10)

        mosaics_saccade = torch.stack(mosaics_saccade).to(device)  # (B, 3, H, W)

        # forward student — vue saccadée
        y_target = torch.stack([x_star, y_star], dim=1).to(device)  # (B, 2)
        z_star = teacher_model(mosaics_saccade)                   # (B, D)

        logits_saccade = z_linear_head(z_star+z_center)                       # (B, 1000)
        pred_saccade   = logits_saccade.argmax(dim=1)
        correct_saccade += (pred_saccade == labels).sum().item()

        total += B

        pbar.set_postfix({
            "acc_center"  : f"{100*correct_center/total:.2f}%",
            "acc_saccade" : f"{100*correct_saccade/total:.2f}%",
        })

        if disp:
            break # !! TEST
        
        if i % 10 == 9:
            acc_center  = 100 * correct_center  / total
            acc_saccade = 100 * correct_saccade / total

            print(f"\n── Résultats ───────────────────────────────────────────────")
            print(f"  Accuracy centre   : {acc_center:.2f}%")
            print(f"  Accuracy saccade  : {acc_saccade:.2f}%")
            print(f"  Delta             : {acc_saccade - acc_center:+.2f}%")

            torch.save({
                "acc_center"  : acc_center,
                "acc_saccade" : acc_saccade,
                "delta"       : acc_saccade - acc_center,
                "n_images"    : total,
                },  "teacher_one_saccade_batch{i+1}.pt")

    return {
        "acc_center"  : acc_center,
        "acc_saccade" : acc_saccade,
        "delta"       : acc_saccade - acc_center,
        "n_images"    : total,
    }




In [23]:
num_workers = 32

# ── lancement ─────────────────────────────────────────────────────────────
results = evaluate_saccade(
    dino_student_y  = dino_student_y,
    z_linear_head   = z_linear_head,
    val_dataset_raw = val_dataset_raw,
    fovea_transform = transform_test,
    zoom            = 1.5,
    grid_size       = 11,
    batch_size      = 12,
    device          = device,
    num_workers     = num_workers
)


Évaluation saccade:   0%|          | 0/4167 [00:05<?, ?it/s]


NameError: name 'mosa01_teacher_one_saccade_estimation' is not defined